In [ ]:
%pip install -U \
  langchain-core \
  langchain-community \
  langchain-openai \
  langchain-google-genai \
  langchain-classic



In [1]:
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# IMPORTANT: on LangChain 1.x, chains live in langchain_classic, not langchain.chains
from langchain_classic.chains.sql_database.query import create_sql_query_chain
# (you could also do: from langchain_classic.chains import create_sql_query_chain)


In [ ]:
#connect to mysql database

host='localhost'
port='3306'
username='sql_username'
password='sql_password'
database_schema='db_name'

mysql_uri=f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"

db=SQLDatabase.from_uri(mysql_uri,sample_rows_in_table_info=2)

In [ ]:
db=SQLDatabase.from_uri(mysql_uri,sample_rows_in_table_info=1)
db.get_table_info()

In [4]:
#create llm prompt template
from langchain_core.prompts import ChatPromptTemplate

template="""Based on the table schema below, write a SQL query that would answer the user's question:
Remember: Only provide me the sql query dont include anything else.Provide me the sql query in a single line dont add line breaks
Table schema: {schema}
Question: {question}
SQL Query:
"""

prompt=ChatPromptTemplate.from_template(template)


In [5]:
def get_schema(db):
    schema=db.get_table_info()
    return schema

In [ ]:
llm=ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    api_key='gemini_api_key')

In [7]:
#create the sql query chain using the llm and the prompt template
sql_chain=(
    RunnablePassthrough.assign(schema=lambda _:get_schema(db))
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)


In [ ]:
#test the sql query chain with a sample question
resp=sql_chain.invoke({"question":"what is the total 'Line Total' for Geiss company"})
print(resp)

In [9]:
#format this query properly
import re
query=re.search(r"```sql\s*(.*?)\s*```",resp,re.DOTALL | re.IGNORECASE)

if query:
    query=query.group(1).strip()

In [10]:
#running the obtained query on the db

db.run(query)

'[(136934.6,)]'

#### RAGAS Implementation

In [ ]:
%pip install ragas
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [ ]:
%pip install langchain-groq
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="llama-3.1-8b-instant",
    api_key="groq_api_key"
)

In [42]:

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm=LangchainLLMWrapper(llm)
evaluator_embeddings=LangchainEmbeddingsWrapper(embeddings)

In [57]:


from ragas import evaluate
from ragas.metrics import AspectCritic, RubricsScore

maliciousness_definition=(
    "Is the response intended to harm, decieve or exploit users?"
)

aspect_critic=AspectCritic(
    name="maliciousness",
    definition=maliciousness_definition,
    llm=evaluator_llm,
)

helpfulness_rubrics = {
    "score1_description": "Response is useless/irrelevant, contains inaccurate/deceptive/misleading information, and/or contains harmful/offensive content. The user would feel not at all satisfied with the content in the response.",
    "score2_description": "Response is minimally relevant to the instruction and may provide some vaguely useful information, but it lacks clarity and detail. It might contain minor inaccuracies. The user would feel only slightly satisfied with the content in the response.",
    "score3_description": "Response is relevant to the instruction and provides some useful content, but could be more relevant, well-defined, comprehensive, and/or detailed. The user would feel somewhat satisfied with the content in the response.",
    "score4_description": "Response is very relevant to the instruction, providing clearly defined information that addresses the instruction's core needs.  It may include additional insights that go slightly beyond the immediate instruction.  The user would feel quite satisfied with the content in the response.",
    "score5_description": "Response is useful and very comprehensive with well-defined key details to address the needs in the instruction and usually beyond what explicitly asked. The user would feel very satisfied with the content in the response.",
}

rubrics_score=RubricsScore(name="helpfulness",rubrics=helpfulness_rubrics,llm=evaluator_llm)

    

In [58]:
from ragas import evaluate
from ragas.metrics import ContextPrecision, Faithfulness

context_precision=ContextPrecision(llm=evaluator_llm)
faithfulness=Faithfulness(llm=evaluator_llm)

In [59]:
import re

user_inputs=[
    "What was the budget of product 12",
    "what are the names of all products in the products table?",
    "list all the customer names from the customers table",
    "what is the name of the customer with customer index=1"
]
responses=[]
for question in user_inputs:
    resp=sql_chain.invoke({"question":question})
    match=re.search(r"```sql\s*(.*?)\s*```",resp,re.DOTALL | re.IGNORECASE)
    if match:
        query=match.group(1).strip()
        responses.append(query)

In [ ]:
print(responses)

In [60]:
references=["SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12';",
            "SELECT `Product Name`ROM products;",
            "SELECT `Customer Names`FROM customers;",
            "SELECT name, state FROM regions;",
            "SELECT `Customer Names` FROM customers WHERE `Customer Index` = 1;"]


In [61]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

In [62]:
n=len(user_inputs)
samples=[]


In [63]:
context=db.get_table_info()
retrieved_contexts=[context]

for i in range(n):
    sample=SingleTurnSample(
        user_input=user_inputs[i],
        retrieved_contexts=list(retrieved_contexts),
        response=responses[i],
        reference=references[i],
    )
    samples.append(sample)
        

In [64]:
ragas_eval_dataset=EvaluationDataset(samples=samples)
ragas_eval_dataset.to_pandas()

,user_input,retrieved_contexts,response,reference
0,What was the budget of product 12,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...
1,what are the names of all products in the prod...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Product Name` FROM products,SELECT `Product Name`ROM products;
2,list all the customer names from the customers...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers,SELECT `Customer Names`FROM customers;
3,what is the name of the customer with customer...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers WHERE `...,"SELECT name, state FROM regions;"


In [65]:
from ragas import evaluate
ragas_metrics=[context_precision,rubrics_score]

result=evaluate(
    metrics=ragas_metrics,
    dataset=ragas_eval_dataset
)
result

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

{'context_precision': 0.2500, 'helpfulness': 3.2500}